# Joint Estimation of GARCH Option Pricing Model

**Author:** Jakob Troger | Financial Econometrics, University of Vienna | January 2026

This notebook reproduces all results and figures from the paper.
All model code lives in `Tools.py` — this notebook is for running and displaying results only.

**To get started:** set `IV_FILE` in the cell below to the path of your IV surface Excel file.


In [ ]:
from Tools import (
    load_all_data,
    HestonNandiGARCH,
    JointEstimator,
    compute_iv_fit,
    print_parameter_table,
    print_iv_fit_summary,
    run_forensic_analysis
)
import numpy as np


## 1. Load Data

Set the path to your implied volatility surface Excel file below.
S&P 500 returns are downloaded automatically from Yahoo Finance.


In [ ]:
# Set the path to your IV surface Excel file
IV_FILE = "SP500_IV_SURFACE.xlsx"  # <-- update this to your file name/path

data = load_all_data(IV_FILE)

returns = data['returns']['log_return'].values
prices  = data['prices']
iv_long = data['iv_long']

print(f"Returns:  {len(returns)} observations")
print(f"IV data:  {len(iv_long)} observations")


## 2. Estimation

### 2.1 Returns-Only Estimation
Baseline MLE using return data only (λ is not identified here).


In [ ]:
model     = HestonNandiGARCH()
estimator = JointEstimator(returns, prices, iv_long, model)

results_returns = estimator.estimate_returns_only()


### 2.2 Joint Estimation
Joint MLE using returns + implied volatility data. This identifies the variance risk premium λ.


In [ ]:
results_joint = estimator.estimate_joint(x0=results_returns['theta'])


## 3. Results

### 3.1 Parameter Estimates (Table 2)


In [ ]:
print_parameter_table(results_returns, results_joint, model)


### 3.2 Implied Volatility Fit (Table 3)


In [ ]:
iv_fit = compute_iv_fit(data, results_joint, model)
print_iv_fit_summary(iv_fit)


## 4. Forensic Analysis

Runs all five experiments explaining *why* the model fails to generate realistic IV skew.
Figures are saved to disk automatically.


In [ ]:
params = results_joint['params']
h0     = results_joint['variance_path'][-1]
S0     = float(prices.iloc[-1])

print(f"Current variance:  h0 = {h0:.6f}  (ann. vol = {np.sqrt(252*h0)*100:.1f}%)")
print(f"Current S&P level: S0 = {S0:.0f}")
print(f"Leverage product:  αγ = {params['alpha']*params['gamma']:.2e}")

run_forensic_analysis(params, h0, S0, save_figures=True)
